In [121]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import sympy as sym
from IPython.display import Math, display
from scipy.linalg import block_diag
from itertools import chain


import config  # noqa: F401  (side effect: puts the project root on sys.path)
from modules import *
from modules.filter_class import Filter
from modules.imu_kinematics_sym import *
from modules.speaker_kinematics_sym import *

sym.init_printing(use_unicode=True)

# Gravity prediction sign inverted

In [8]:
gravity_sym = sym.Symbol(r"g", real=True, positive=True)
gravity_vector_sym = sym.Matrix([0, 0, -1 * gravity_sym])

q = np.array([0.9996, 0.02769, 0.01076, -0.0000006])
g_h = -R(q).T @ np.array([0, 0, -9.81])
g_h

array([-0.21102708,  0.54306036,  9.79268508])

In [9]:
g_h_old = R(q).T @ np.array([0, 0, -9.81])
print(f"{g_h_old} <- aktuell")

g_h_no_transpose = R(q) @ np.array([0, 0, -9.81])
print(f"{g_h_no_transpose} <- kein transpose")

g_h_sign_reversed = -R(q).T @ np.array([0, 0, -9.81])
print(f"{g_h_sign_reversed} <- -R(q)^T")

g_h_both = -R(q) @ np.array([0, 0, -9.81])
print(f"{g_h_both} <- beides")

[ 0.21102708 -0.54306036 -9.79268508] <- aktuell
[-0.21102643  0.54306062 -9.79268508] <- kein transpose
[-0.21102708  0.54306036  9.79268508] <- -R(q)^T
[ 0.21102643 -0.54306062  9.79268508] <- beides


# Constructing Matrices

## The main two variables that determine Matrix size/layout/content

1. number of speakers $N$

2. which measurement $y$? ($y_{g}, y_{p_{s}q_{s}}, y_{p}, y_{\psi}$)

### 1. number of speakers

Extract it from the code using the length of the state `self.x` of the Filter class instance.

```
N = (len(self.x) - get_def("imu nominal state")["len"]) / get_def("speaker nominal state")["len"]
```

In [153]:
N = 3

esekf = Filter()
esekf.x = np.zeros(get_def("imu nominal state")["len"])
# position and velocity start out at zero
esekf.x[get_def("imu orientation")["slice"]] = [1, 0, 0, 0]

# display(esekf.P), display(esekf.Q), display(esekf.J), display(esekf.W)
display(esekf.x)
# F, H, I and K get constructed at runtime

array([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

# Easy to hard

## Identity matrix

is used in the propagation of the covariance at the end of the correction step

In [36]:
esekf.I = np.eye(
    get_def("imu error state")["len"] + N * get_def("speaker error state")["len"]
)

# or
# esekf.I = np.eye(len(delta_x))
# but delta_x only gets calculated during the correction step,
# so I have to see how I want to solve this
esekf.I.shape

## Process Noise Covariance Q

In [128]:
N = 0
# how it was calculated so far

# variance caluclations
variance_accel = ((g2ms2(230 * 1e-6) * np.sqrt(IMU_SAMPLE_FREQ)) ** 2) * (
    (1 / IMU_SAMPLE_FREQ) ** 2
)
variance_gyro = ((np.deg2rad(0.015) * np.sqrt(IMU_SAMPLE_FREQ)) ** 2) * (
    (1 / IMU_SAMPLE_FREQ) ** 2
)

imu_process_noise_j = {
    "velocity": variance_accel,
    "orientation": variance_gyro,
    "accel_bias": 9e-9,  # guess
    "gyro_bias": 4e-12,  # guess
}

old_Q = np.diag(np.repeat(list(imu_process_noise_j.values()), 3))

print(np.array_equal(old_Q, esekf.Q), old_Q.shape)

# now I also have to add the way these elements stack diagonally with the speaker

imu_process_noise_j = {
    "velocity": [variance_accel] * 3,
    "orientation": [variance_gyro] * 3,
    "accel_bias": [9e-9] * 3,  # guess
    "gyro_bias": [4e-12] * 3,  # guess
}

speaker_process_noise_j = {
    "velocity": [1e-8] * 3,  # guess
    "orientation": [1e-8] * 3,  # guess
}

imu_array = list(chain(*imu_process_noise_j.values()))
speaker_array = list(chain(*speaker_process_noise_j.values()))

whole_array = imu_array + speaker_array * N
print(whole_array)

# whole_array
new_Q = np.diag(whole_array)

print(new_Q.shape)
print(np.array_equal(new_Q, esekf.Q))

# diese noise Werte sind aktuell noch sehr statisch.
# Man könnte überlegen es so zu machen, das man die IMU_SAMPLE_FREQ im nachhinein noch ändern kann
# Dann wäre das aber auch sinnvoll, es für die guesstimates auch zu machen
# deswegen fürs erste würde ich einfach sagen bleibt bei statischen Werten,
# schließlich können wir ja schon davon ausgehen das die sample rate ungefähr gleich bleibt
# oder ich definiere es halt ca so

ACCEL_NSD_MU_G = 230


def var_accel(imu_sample_freq=IMU_SAMPLE_FREQ):
    accelNSD = g2ms2(ACCEL_NSD_MU_G * 1e-6)
    sigma_accel = accelNSD * np.sqrt(imu_sample_freq)
    variance = (sigma_accel**2) * ((1 / IMU_SAMPLE_FREQ) ** 2)
    return variance


temp_variance_accel = var_accel()

temp_variance_accel

# und dann kann ich halt mit wenig extra Aufwand später noch das alles so umschreiben,
# das es in Abhängigkeit von der IMU_SAMPLE_FREQ ist

True (12, 12)
[np.float64(4.915375193594443e-08), np.float64(4.915375193594443e-08), np.float64(4.915375193594443e-08), np.float64(6.622117821450184e-10), np.float64(6.622117821450184e-10), np.float64(6.622117821450184e-10), 9e-09, 9e-09, 9e-09, 4e-12, 4e-12, 4e-12]
(12, 12)
True


## Process Noise Jacobian J

ist ja definiert als

$$J = \left. \dfrac{\partial f}{\partial j} \right|_{x, u}$$

oder für einen Speaker spezifisch

$$J = \left. \frac{\partial f}{\partial j} \right|_{x, u} = 
\begin{bmatrix}
0 & 0 & 0 & 0 & 0 & 0 \\
I & 0 & 0 & 0 & 0 & 0 \\
0 & I & 0 & 0 & 0 & 0 \\
0 & 0 & I & 0 & 0 & 0 \\
0 & 0 & 0 & I & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & I & 0 \\
0 & 0 & 0 & 0 & 0 & I
\end{bmatrix}$$

In [ ]:
N = 1
# bisher so
old_J = np.vstack((np.zeros((3, 3 * 4)), np.eye(3 * 4)))

print(np.array_equal(old_J, esekf.J), old_J.shape)
print(old_J)
# die IMU wird immer so aussehen


# mit speaker (und etwas sleeker bei np.eye), als Funktion gedacht
def build_J(n=N):
    # dieser Teil könnte sogar noch einen Schritt weiter gedacht werden,
    # wenn ich hier actually die jacobians konstruiere von \partial f / \partial j
    # das würde es sehr future proof machen, wenn Ich Sachen hinzufüge oder wegnehme
    # nur wenn ich die Reiehnfolge ändern würde, dann wäre es nicht ganz ideal
    # aber selbst da sollte es theoretisch machbar sein, schließlich ist diese Matrix ja dazu da die Sachen an die richtige Stelle zuzuordnene
    imu_J = np.eye(15, 12, k=-3)
    speaker_J = np.eye(9, 6, k=-3)

    return block_diag(old_J, *[speaker_J] * N)


np.set_printoptions(linewidth=150)
build_J()

## Error State Covariance P

wie bei den anderen auch, bis auf die Sache das ich die Varianzwerte probably im vector_defs.py definieren sollte und auch das multiplizieren mit (meistens) 3 entspricht ja auch der Länge des entsprechenden error states. Aber ansonsten wird so der State zusammen gebaut

In [158]:
N = 3
# bisher
initial_certainties = {
    "position": 1e-3,  # guess
    "velocity": 1e-3,  # guess
    "orientation": 5e-3,  # guess
    "accel_bias": 0.06,  # guess
    "gyro_bias": 2e-3,  # guess
}

old_P = np.diag(
    np.repeat([sigma_sqrd for _, sigma_sqrd in initial_certainties.items()], 3)
)

print(np.array_equal(old_P, esekf.P), old_P.shape)

# neuer Ansatz
imu_initial_variances = {
    "position": [1e-3] * 3,  # guess
    "velocity": [1e-3] * 3,  # guess
    "orientation": [5e-3] * 3,  # guess
    "accel_bias": [0.06] * 3,  # guess
    "gyro_bias": [2e-3] * 3,  # guess
}

speaker_initial_variances = {
    "position": [1e-4] * 3,  # guess
    "velocity": [1e-5] * 3,  # guess
    "orientation": [1e-6] * 3,  # guess
}

imu_variances = list(chain(*imu_initial_variances.values()))
speaker_variances = list(chain(*speaker_initial_variances.values()))

np.set_printoptions(linewidth=150, precision=10)
new_P = np.diag(imu_variances + speaker_variances * N)
new_P

True (15, 15)


array([[0.001   , 0.      , 0.      , ..., 0.      , 0.      , 0.      ],
       [0.      , 0.001   , 0.      , ..., 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.001   , ..., 0.      , 0.      , 0.      ],
       ...,
       [0.      , 0.      , 0.      , ..., 0.000001, 0.      , 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 0.000001, 0.      ],
       [0.      , 0.      , 0.      , ..., 0.      , 0.      , 0.000001]], shape=(42, 42))

# Error State Jacobian F

hier muss ich vorab noch ein paar Experimente machen, glaube aber das das auch selbstredend sein wird

Ich habe ja in den kinematics_sym Dateien schon die Funktionen alle gesammelt, jetzt muss ich die nur noch zusammenbauen zu einer großen Funktion $f$, und dann davon halt die Jacobimatrix bilden. Und ich glaube da wird schnell das Muster klar, das ich es auch wieder mit scipy.block_diag lösen kann, ungefähr so

$$
F_{\delta x} = \left. \frac{\partial f_{\delta x}}{\partial \delta x} \right|_{x, u} = 
\begin{bmatrix}
\color{green}{\dfrac{\partial f_{\delta x_\text{IMU}}}{\partial \delta x_{\text{IMU}}}} & \mathbf{0}_{15 \times 9} & \mathbf{0}_{15 \times 9} & \cdots \\
\mathbf{0}_{9 \times 15} & \dfrac{\partial f_{\delta x_{i=1}}}{\partial \delta x_{i=1}} & \mathbf{0}_{9 \times 9} & \cdots \\
\mathbf{0}_{9 \times 15} & \mathbf{0}_{9 \times 9} & \dfrac{\partial f_{\delta x_{i=2}}}{\partial \delta x_{i=2}} & \cdots \\
\vdots & \vdots & \vdots & \ddots
\end{bmatrix}
$$

## Measurement Noise Covariance W

hier ist es zum ersten mal wichtig, which measurement $y$? ($y_{g}, y_{p_{s}q_{s}}, y_{p}, y_{\psi}$)

Hierzu ist gerade noch das größte Hindernis, das mein vector_defs.py da doch noch nicht komplett sauber ausgearbeitet ist. Das müsste ich vorab machen, bevor ich mit diesen Sachen weitermachen kann

In [ ]:
# bisher
measurement_noise = {
    "accel_gravity": 2.5e-3,  # guess
}

old_W = np.diag(
    np.repeat([sigma_sqrd for _, sigma_sqrd in measurement_noise.items()], 3)
)

# zwischen Verbesserung
measurement_noise = {
    "accel_gravity": [2.5e-3] * 3,  # guess
    "face_detection_position": [2.5e-4] * 3,  # guess
    "face_detection_orientation": [2.5e-4] * 3,  # guess
    "fake_global_position": [2.5e-5] * 3,  # guess
    "fake_imu_yaw": [5e-6] * 1,  # guess
}

new_W = np.diag(
    list(chain(measurement_noise["accel_gravity"], measurement_noise["fake_imu_yaw"]))
)

np.set_printoptions(linewidth=150, precision=6)
esekf.W, old_W, new_W

(array([[0.0025, 0.    , 0.    ],
        [0.    , 0.0025, 0.    ],
        [0.    , 0.    , 0.0025]]),
 array([[0.0025, 0.    , 0.    ],
        [0.    , 0.0025, 0.    ],
        [0.    , 0.    , 0.0025]]),
 array([[0.0025  , 0.      , 0.      , 0.      ],
        [0.      , 0.0025  , 0.      , 0.      ],
        [0.      , 0.      , 0.0025  , 0.      ],
        [0.      , 0.      , 0.      , 0.000005]]))